# 🔀 Conditional Workflow — Solution

Complete solution for the expense claim triage workflow, including the optional
4th escalation path for high-value claims.

**Pipeline:**
```
[Expense Claim]
    → PolicyCheckerAgent
    → to_policy_decision
    → route_claim  (3-way: approved / clarification_needed / rejected)
       ├─ to_approval_agent  → ApprovalDrafterAgent
       ├─ to_clarification   → (terminal output)
       └─ to_rejection       → (terminal output)
```

In [ ]:
####### Linux / MacOS Setup Instructions #######
# ! sudo apt update && sudo apt install graphviz -y   # Linux
# ! brew install graphviz                              # macOS

In [ ]:
# ! pip install -r ../../../Installation/requirements.txt -U

In [ ]:
import os
from typing import cast
from dataclasses import dataclass
from typing_extensions import Literal
from pydantic import BaseModel
from IPython.display import display, SVG, HTML

In [ ]:
from azure.identity.aio import AzureCliCredential
from dotenv import load_dotenv

from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from agent_framework import (
    AgentExecutor,
    AgentResponse,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowEvent,
    executor,
    WorkflowViz
)

In [ ]:
load_dotenv()

In [ ]:
# SOLUTION — Task 1: Agent Instructions

PolicyCheckerInstructions = """
You are an expense policy compliance officer at AtlasTech Maroc, Casablanca.

When you receive an expense claim, check it against these company rules:
  - Domestic per diem (meals + incidentals): max 600 MAD/day
  - Hotel (Morocco): max 800 MAD/night
  - Submission deadline: within 5 business days of trip end date
  - International per diem: max 1,800 MAD/day

Classify the claim as exactly one of:
  - "approved": all items comply with policy
  - "clarification_needed": information is missing or ambiguous (e.g. no receipts mentioned, dates unclear)
  - "rejected": one or more items clearly exceed policy limits

Return ONLY a JSON object with these fields — no other text:
{
  "status": "approved" | "clarification_needed" | "rejected",
  "reason": "<specific explanation citing the rule violated or confirmed>",
  "claim_summary": "<one-line summary: employee, trip, total amount>"
}
"""

ApprovalDrafterInstructions = """
You are the finance team assistant at AtlasTech Maroc.
When you receive details of an approved expense claim, draft a short professional
approval notification email to the employee.

The email must:
  - Confirm the claim is approved
  - Include a one-line summary of what was approved
  - State that payment will be processed within 5 working days
  - Be friendly and professional
"""

In [ ]:
# Sample expense claim — hotel (1,050 MAD/night) exceeds the 800 MAD policy limit → rejected
EXPENSE_CLAIM = """
Employee: Karim Alaoui, Senior Engineer
Trip: Client on-site visit, Marrakech, 14–16 March 2026 (2 nights)
Submission date: 20 March 2026

Expenses:
  - Hotel: 2 nights × 1,050 MAD = 2,100 MAD  (Hotel La Mamounia)
  - Meals & incidentals: 2 days × 590 MAD   = 1,180 MAD
  - Taxi Casablanca → airport              =   150 MAD

Total claimed: 3,430 MAD
"""

In [ ]:
# SOLUTION — Task 2: Output Models

class PolicyCheckerOutput(BaseModel):
    status: Literal["approved", "clarification_needed", "rejected"]
    reason: str
    claim_summary: str

@dataclass
class ClaimDecision:
    status: str
    reason: str
    claim_summary: str

In [ ]:
# SOLUTION — Task 3: Policy Decision Executor

@executor(id="to_policy_decision")
async def to_policy_decision(response: AgentExecutorResponse, ctx: WorkflowContext[ClaimDecision]) -> None:
    response_text = response.agent_response.text.strip()
    print(f"Policy checker raw response: {response_text}")

    parsed = PolicyCheckerOutput.model_validate_json(response_text)
    await ctx.send_message(
        ClaimDecision(
            status=parsed.status,
            reason=parsed.reason,
            claim_summary=parsed.claim_summary,
        )
    )


# SOLUTION — Task 4: 3-Way Routing Function

def route_claim(decision: ClaimDecision, target_ids: list[str]) -> list[str]:
    approval_id, clarification_id, rejection_id = target_ids
    if decision.status == "approved":
        return [approval_id]
    elif decision.status == "clarification_needed":
        return [clarification_id]
    else:
        return [rejection_id]


# Provided executors (unchanged from challenge)

@executor(id="to_approval_agent")
async def to_approval_agent(decision: ClaimDecision, ctx: WorkflowContext) -> None:
    await ctx.send_message(
        AgentExecutorRequest(
            messages=[Message("user", text=(
                f"The following expense claim has been approved:\n"
                f"Summary: {decision.claim_summary}\n"
                f"Notes: {decision.reason}\n\n"
                f"Please draft the approval notification email."
            ))],
            should_respond=True
        )
    )


@executor(id="to_clarification")
async def to_clarification(decision: ClaimDecision, ctx: WorkflowContext) -> None:
    await ctx.yield_output(
        f"⚠️ Clarification needed for claim '{decision.claim_summary}':\n{decision.reason}"
    )


@executor(id="to_rejection")
async def to_rejection(decision: ClaimDecision, ctx: WorkflowContext) -> None:
    await ctx.yield_output(
        f"❌ Expense claim rejected: '{decision.claim_summary}'\nReason: {decision.reason}"
    )

In [ ]:
async with (
    AzureCliCredential() as credential,
    AzureAIAgentsProvider(credential=credential) as provider,
):
    policy_agent_obj = None
    approval_agent_obj = None
    try:
        policy_agent_obj = await provider.create_agent(
            name="policy-checker-agent",
            instructions=PolicyCheckerInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        policy_checker_agent = AgentExecutor(policy_agent_obj, id="policy_checker_agent")

        approval_agent_obj = await provider.create_agent(
            name="approval-drafter-agent",
            instructions=ApprovalDrafterInstructions,
            model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        )
        approval_agent = AgentExecutor(approval_agent_obj, id="approval_agent")

        # SOLUTION — Task 5: Build the workflow
        workflow = (
            WorkflowBuilder(start_executor=policy_checker_agent)
                .add_edge(policy_checker_agent, to_policy_decision)
                .add_multi_selection_edge_group(
                    to_policy_decision,
                    [to_approval_agent, to_clarification, to_rejection],
                    selection_func=route_claim,
                )
                .add_edge(to_approval_agent, approval_agent)
                .build()
        )

        # Visualise
        print("Workflow visualization:")
        viz = WorkflowViz(workflow)
        print(viz.to_mermaid())
        svg_file = viz.export(format="svg")
        if svg_file and os.path.exists(svg_file):
            try:
                display(SVG(filename=svg_file))
            except Exception:
                with open(svg_file, "r", encoding="utf-8") as f:
                    display(HTML(f.read()))

        task = (
            "Please review the following expense claim submitted by an AtlasTech Maroc employee "
            "and check whether it complies with company policy.\n\n" + EXPENSE_CLAIM
        )

        events = await workflow.run(task)

        # outputs can be plain strings (from yield_output) or AgentResponse objects
        for output in events.get_outputs():
            if isinstance(output, str):
                print(output)
            else:
                print(f"{output.messages[0].author_name}: {output.text}\n")

        print("Final state:", events.get_final_state())

    finally:
        for agent_obj in [policy_agent_obj, approval_agent_obj]:
            if agent_obj is not None:
                try:
                    await provider._agents_client.delete_agent(agent_obj.id)
                except Exception:
                    pass
        print("done")

---

## ⭐ OPTIONAL SOLUTION: 4th Routing Path — Manager Escalation

Below is the complete solution with the 4th `"escalated"` path added.
Swap `EXPENSE_CLAIM` for a claim exceeding 5,000 MAD to trigger it.

In [ ]:
# OPTIONAL SOLUTION — extended PolicyCheckerOutput with escalation

class PolicyCheckerOutputV2(BaseModel):
    status: Literal["approved", "clarification_needed", "rejected", "escalated"]
    reason: str
    claim_summary: str


PolicyCheckerInstructionsV2 = PolicyCheckerInstructions + """
  - "escalated": the claim is policy-compliant but the total exceeds 5,000 MAD,
    requiring manager sign-off before approval.
"""


@executor(id="to_escalation")
async def to_escalation(decision: ClaimDecision, ctx: WorkflowContext) -> None:
    await ctx.yield_output(
        f"🔺 Claim escalated to manager: '{decision.claim_summary}'\n"
        f"Reason: {decision.reason}\n"
        f"Action required: manager approval before processing."
    )


def route_claim_v2(decision: ClaimDecision, target_ids: list[str]) -> list[str]:
    approval_id, clarification_id, rejection_id, escalation_id = target_ids
    if decision.status == "approved":
        return [approval_id]
    elif decision.status == "clarification_needed":
        return [clarification_id]
    elif decision.status == "escalated":
        return [escalation_id]
    else:
        return [rejection_id]


# Build the extended workflow (same agent creation as above)
# workflow_v2 = (
#     WorkflowBuilder(start_executor=policy_checker_agent)
#         .add_edge(policy_checker_agent, to_policy_decision)
#         .add_multi_selection_edge_group(
#             to_policy_decision,
#             [to_approval_agent, to_clarification, to_rejection, to_escalation],
#             selection_func=route_claim_v2,
#         )
#         .add_edge(to_approval_agent, approval_agent)
#         .build()
# )